# 07: RAFT + geometria v2 i demo (koło + próbka ze zbioru)

**Geometria v2** (`src/raft_geom.py`, `correct_ab`) poprawia dwa błędy na krawędziach ruchomych obiektów:
1. na krawędzi obiektu wybiera obiekt, a nie tło (wcześniej obok obiektu zostawał pasek tła);
2. miejsca, których kamera A nie widziała (tło zasłonięte w chwili odczytu wiersza), bierze z kamery B.
RAFT liczy przepływ w obie strony (A→B i B→A), więc korekcja trwa ok. 2× dłużej.

Kolejność: setup → zbiór → ocena (v1 vs v2) → dostrojenie RAFT z geometrią v2 → **demo**.
Samo demo (sekcja 4) działa też bez sekcji 1–3: weźmie najnowszy dostrojony model z Drive.

In [ ]:
# === SETUP: uruchamiaj zawsze jako pierwszą komórkę ===
# Kod jest pobierany z GitHuba na maszynę Colaba (lokalne pliki z VS Code tam nie trafiają).
# Po zmianie kodu: commit + push w VS Code, potem ponownie ta komórka (robi git pull).
REPO = 'kordiusz/RollingShutter'   # repozytorium na GitHubie
BRANCH = 'raft-geom-v2'   # po scaleniu z main: 'main'

import os, sys, subprocess, getpass, importlib
from google.colab import drive
drive.mount('/content/drive')

TOKEN_FILE = '/content/drive/MyDrive/rs_project/.github_token'   # opcjonalnie: token zapisany na Drive
CODE = '/content/code'

def _git(*args):
    r = subprocess.run(['git', *args], capture_output=True, text=True)
    if r.returncode != 0:
        msg = (r.stderr or r.stdout).replace(token, '***') if 'token' in globals() else r.stderr
        raise RuntimeError('git ' + args[0] + ' nie powiodło się:\n' + msg)
    return r.stdout

if not os.path.exists(CODE):
    # repozytorium jest publiczne: token potrzebny tylko, gdy jest zapisany na Drive (np. repo prywatne)
    token = open(TOKEN_FILE).read().strip() if os.path.exists(TOKEN_FILE) else ''
    auth = f'{token}@' if token else ''
    _git('clone', '-q', '-b', BRANCH, f'https://{auth}github.com/{REPO}.git', CODE)
else:
    _git('-C', CODE, 'fetch', '-q', 'origin', BRANCH)
    _git('-C', CODE, 'checkout', '-q', BRANCH)
    _git('-C', CODE, 'pull', '-q', 'origin', BRANCH)
os.chdir(CODE)
sys.path.insert(0, f'{CODE}/src')
import project_paths as P
importlib.reload(P)
P.ensure_dirs()
print(_git('log', '-1', '--format=kod: %h  %s  (%cd)', '--date=short'))
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1. Zbiór danych (rozpakowanie z Drive na dysk maszyny Colaba)

In [ ]:
import os, glob
DATASET_NAME = 'rs_video_x4k'
z = P.find_dataset_zip(DATASET_NAME)
assert z, f'Brak {DATASET_NAME}.zip w {P.DATASETS} - wrzuć tam spakowany folder data/{DATASET_NAME}'
UNZ = f'/content/unz_{DATASET_NAME}'
if not os.path.exists(UNZ):
    print('rozpakowuję', z, '...')
    !unzip -q "{z}" -d "{UNZ}"
# folder z podfolderami train/ i val/ (niezależnie od tego, jak spakowano)
DATA = next(os.path.dirname(p) for p in sorted(glob.glob(f'{UNZ}/**/train', recursive=True)))
print('DATA =', DATA, '|', len(os.listdir(f'{DATA}/train')), 'train /', len(os.listdir(f'{DATA}/val')), 'val')
WORKERS = min(8, os.cpu_count())

## 2. Ocena: stara (v1) i nowa (v2) geometria
Ten sam RAFT, różni się tylko geometria. `psnr` = wynik, `psnr_in` = wejście RS A (bez korekcji).

In [ ]:
import json, pandas as pd
from datetime import date
CK_OLD = P.find_run('raft_geom')            # dostrojony wcześniej RAFT (z geometrią v1), jeśli jest
print('dostrojony RAFT:', CK_OLD)
rows = []
for label, ck, geom in [('gotowy RAFT, v1', None, 'v1'), ('gotowy RAFT, v2', None, 'v2'),
                        ('dostrojony RAFT, v1', CK_OLD, 'v1'), ('dostrojony RAFT, v2', CK_OLD, 'v2')]:
    if ck is None and 'dostrojony' in label:
        continue
    ej = f'/content/eval_{label.replace(" ", "_").replace(",", "")}.json'
    src = f'--eval "{ck}"' if ck else '--zero-shot'
    !python src/train_raft.py --data "{DATA}" {src} --geom {geom} --eval-json "{ej}" --bs 8 --workers {WORKERS} --out /content/runs_tmp --name tmp > /dev/null
    m = json.load(open(ej))['metrics']
    rows.append({'model': label, **{k: round(v, 3) for k, v in m.items()}})
display(pd.DataFrame(rows).set_index('model'))

## 3. Dostrojenie RAFT z geometrią v2
Startuje z dostrojonego wcześniej RAFT (jeśli jest), 5 epok. Wynik: `runs/<data>_raft_geom_v2/best.pt` na Drive.

In [ ]:
RUN = f'{date.today()}_raft_geom_v2'
init = f'--init "{CK_OLD}"' if CK_OLD else ''
!python src/train_raft.py --data "{DATA}" {init} --geom v2 --epochs 5 --bs 8 --lr 1e-5 --workers {WORKERS} --out "{P.RUNS}" --name {RUN}
from IPython.display import Image, display
p = sorted((P.RUNS / RUN).glob('preview_ep*.png'))[-1]
print(p); display(Image(str(p)))

## 4. DEMO
Wejście: folder próbki (`rs_a.png`, `rs_b.png` tak jak zapisuje kamera B, `meta.json` z czasem odczytu,
sync i bazą; `gs_a.png` jeśli znamy wzorzec). W repo są dwa przykłady:
- `examples/kolo`: koło w kratkę jadące w prawo 2 px/ms (`src/synth_shapes.py`), odczyt 20 ms;
- `examples/x4k_val_00273_02`: mocno wykrzywiona próbka ze zbioru walidacyjnego (torba w zygzaki).

Wynik: obraz (RS A | RS B | wynik | wzorzec | stara geometria) i animacja przed/po.

In [ ]:
from IPython.display import Image, display
CKPT = P.find_run('raft_geom_v2') or P.find_run('raft_geom')   # najnowszy dostrojony; None = gotowy RAFT
print('model:', CKPT or 'gotowy RAFT (bez dostrajania)')
OUT = P.RESULTS / 'demo_v2'; OUT.mkdir(parents=True, exist_ok=True)
ck = f'--ckpt "{CKPT}"' if CKPT else ''
!python src/demo.py examples/kolo examples/x4k_val_00273_02 {ck} --compare-v1 --out "{OUT}/demo.png" --gif "{OUT}/demo.gif"
display(Image(str(OUT / 'demo.png')))
display(Image(str(OUT / 'demo.gif')))

### Własne obrazy
Wrzuć dwa obrazy 256×256 (RS z kamery A i z kamery B, B obrócony tak jak zapisuje go kamera)
i ustaw czas odczytu, sync i bazę. Inne koło wygenerujesz np. tak:
`!python src/synth_shapes.py --out /content/moje_kolo --vx 0 --vy 3` (koło w dół, 3 px/ms).

In [ ]:
from google.colab import files
up = files.upload()                       # najpierw RS A, potem RS B
names = list(up)
READOUT_MS, SYNC_MS, BASELINE = 20.0, 0.0, 0.0
!python src/demo.py --a "{names[0]}" --b "{names[1]}" --readout-ms {READOUT_MS} --sync-ms {SYNC_MS} --baseline {BASELINE} {ck} --out /content/moje.png
display(Image('/content/moje.png'))